# Phase 2: GNN Architecture Comparison — Drug-Molecule-Property-Prediction
**Date:** 2026-04-07  
**Researcher:** Anthony Rodrigues  
**Dataset:** ogbg-molhiv (41,127 molecules, scaffold split)  
**Primary Metric:** ROC-AUC (OGB standard)

## Research Question
Phase 1 established that traditional ML (RF+Combined features) achieves ROC-AUC 0.7707 (Mark's CatBoost: 0.7782). SOTA DeeperGCN achieves 0.8476. **Can graph neural networks close this gap?**

Since Mark covered tree-based models and fingerprint engineering in Phase 1, Phase 2 focuses on GNN architectures — the approaches that can directly model molecular graph topology (bonds, rings, substructures) that fingerprints only approximate.

**Architectures tested:**
1. **GCN** — Graph Convolutional Network (Kipf & Welling 2017) — the classic baseline
2. **GIN** — Graph Isomorphism Network (Xu et al. 2019) — theoretically most powerful for graph classification  
3. **GAT** — Graph Attention Network (Veličković et al. 2018) — attention-weighted neighbor aggregation  
4. **GraphSAGE** — Hamilton et al. 2017 — mean/max sampling aggregation

**Research references:**
- Xu et al., 2019: GIN achieves Weisfeiler-Lehman test-level expressivity — the theoretical upper bound for message-passing GNNs
- Hu et al., 2020 (OGB paper): GIN + virtual node reaches 0.7558 AUC on ogbg-molhiv
- Corso et al., 2020 (PNA): Multi-aggregator GNNs improve over single-aggregator by ~0.01-0.03 AUC

In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import DataLoader
from torch_geometric.datasets import MoleculeNet
from torch_geometric.nn import (
    GCNConv, GINConv, GATConv, SAGEConv,
    global_mean_pool, global_add_pool, global_max_pool
)
from sklearn.metrics import roc_auc_score
from collections import defaultdict

# Paths
BASE_DIR = Path('/Users/anthonyrodrigues/Desktop/YC-Portfolio-Projects/Drug-Molecule-Property-Prediction')
DATA_DIR = BASE_DIR / 'data' / 'raw'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device('cpu')  # MPS has overhead for PyG scatter ops
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Hyperparameters
BATCH_SIZE = 128
HIDDEN_DIM = 128
NUM_LAYERS = 3
DROPOUT = 0.5
LR = 1e-3
EPOCHS = 50
PATIENCE = 12  # early stopping

In [ ]:
# Load PyG MoleculeNet HIV dataset
dataset = MoleculeNet(root=DATA_DIR / 'pyg', name='HIV')
print(f'Total molecules: {len(dataset)}')
print(f'Node features: {dataset.num_features}')
print(f'Sample data: {dataset[0]}')

# Load OGB scaffold split (CSV-based, avoids torch.load compatibility issue)
mol_df = pd.read_csv(DATA_DIR / 'ogbg_molhiv' / 'mapping' / 'mol.csv.gz')
train_ogb = pd.read_csv(DATA_DIR / 'ogbg_molhiv' / 'split' / 'scaffold' / 'train.csv.gz', header=None)[0].tolist()
val_ogb   = pd.read_csv(DATA_DIR / 'ogbg_molhiv' / 'split' / 'scaffold' / 'valid.csv.gz', header=None)[0].tolist()
test_ogb  = pd.read_csv(DATA_DIR / 'ogbg_molhiv' / 'split' / 'scaffold' / 'test.csv.gz',  header=None)[0].tolist()

# Build SMILES -> PyG index map
pyg_smiles_to_idx = {dataset[i].smiles: i for i in range(len(dataset))}

def map_ogb_to_pyg(ogb_indices, mol_df, smiles_map):
    pyg_idx = []
    for i in ogb_indices:
        smi = mol_df.iloc[i]['smiles']
        if smi in smiles_map:
            pyg_idx.append(smiles_map[smi])
    return pyg_idx

train_idx = map_ogb_to_pyg(train_ogb, mol_df, pyg_smiles_to_idx)
val_idx   = map_ogb_to_pyg(val_ogb,   mol_df, pyg_smiles_to_idx)
test_idx  = map_ogb_to_pyg(test_ogb,  mol_df, pyg_smiles_to_idx)

print(f'Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}')
print(f'Alignment: {len(train_idx)+len(val_idx)+len(test_idx):,} / {len(dataset):,} mapped')

# Compute class balance
train_labels = [dataset[i].y.item() for i in train_idx]
test_labels  = [dataset[i].y.item() for i in test_idx]
print(f'Train positive rate: {np.mean(train_labels)*100:.2f}%')
print(f'Test positive rate:  {np.mean(test_labels)*100:.2f}%')

In [ ]:
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader as PyGDataLoader

# Build split datasets
train_data = [dataset[i] for i in train_idx]
val_data   = [dataset[i] for i in val_idx]
test_data  = [dataset[i] for i in test_idx]

train_loader = PyGDataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = PyGDataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = PyGDataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

# Input dim
IN_DIM = dataset.num_features
print(f'Input feature dim: {IN_DIM}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution per split
splits_info = {
    'Train': train_labels,
    'Val': [dataset[i].y.item() for i in val_idx],
    'Test': test_labels
}
split_names = list(splits_info.keys())
pos_rates = [np.mean(v)*100 for v in splits_info.values()]
neg_rates = [100 - p for p in pos_rates]
x = np.arange(len(split_names))
bars1 = axes[0].bar(x, neg_rates, label='Inactive (0)', color='#4472C4', alpha=0.8)
bars2 = axes[0].bar(x, pos_rates, bottom=neg_rates, label='Active HIV (1)', color='#ED7D31', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(split_names, fontsize=12)
axes[0].set_ylabel('Percentage (%)', fontsize=12)
axes[0].set_title('Class Distribution by Split\n(Scaffold Split)', fontsize=13, fontweight='bold')
axes[0].legend()
for i, p in enumerate(pos_rates):
    axes[0].text(i, 50, f'{p:.1f}%\nactive', ha='center', fontsize=10, color='white', fontweight='bold')

# Molecule size distribution (heavy atom count)
atom_counts = [dataset[i].x.shape[0] for i in train_idx[:3000]]
axes[1].hist(atom_counts, bins=40, color='#70AD47', edgecolor='white', alpha=0.85)
axes[1].axvline(np.mean(atom_counts), color='red', linestyle='--', linewidth=2, label=f'Mean={np.mean(atom_counts):.1f}')
axes[1].set_xlabel('Atoms per Molecule', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Molecule Size Distribution\n(Sample of 3000 training molecules)', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('ogbg-molhiv Dataset Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'phase2_dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: phase2_dataset_overview.png')

## GNN Architecture Definitions

Four architectures, progressing from simple to expressive:

| Model | Aggregation | Key Property |
|-------|-------------|--------------|
| GCN | Sum (normalized) | Simple, fast, renormalization trick |
| GIN | Sum (MLP) | WL-test equivalent — maximally discriminative |
| GAT | Attention-weighted sum | Learns which neighbors matter more |
| GraphSAGE | Mean + concat | Inductive, generalizes to unseen graphs |

All use: 3 layers → BatchNorm → Dropout(0.5) → Global Mean Pool → 2-layer MLP classifier

In [ ]:
class GCN(nn.Module):
    """Graph Convolutional Network — Kipf & Welling 2017."""
    def __init__(self, in_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.lin1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = nn.Linear(hidden_dim // 2, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x).squeeze(-1)


class GIN(nn.Module):
    """Graph Isomorphism Network — Xu et al. 2019.
    WL-test equivalent: most powerful simple GNN for graph classification."""
    def __init__(self, in_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        mlp0 = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
                              nn.Linear(hidden_dim, hidden_dim))
        self.convs.append(GINConv(mlp0, train_eps=True))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            mlp = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
                                 nn.Linear(hidden_dim, hidden_dim))
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.lin1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = nn.Linear(hidden_dim // 2, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x).squeeze(-1)


class GAT(nn.Module):
    """Graph Attention Network — Veličković et al. 2018.
    Learns attention weights over neighbors — which atoms matter more."""
    def __init__(self, in_dim, hidden_dim, num_layers, dropout, heads=4):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        # First layer: in_dim → hidden_dim (output: hidden_dim)
        self.convs.append(GATConv(in_dim, hidden_dim // heads, heads=heads, dropout=dropout, concat=True))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GATConv(hidden_dim, hidden_dim // heads, heads=heads, dropout=dropout, concat=True))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.lin1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = nn.Linear(hidden_dim // 2, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x).squeeze(-1)


class GraphSAGE(nn.Module):
    """GraphSAGE — Hamilton et al. 2017.
    Mean aggregation + skip connections — inductive learning approach."""
    def __init__(self, in_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.lin1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = nn.Linear(hidden_dim // 2, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x).squeeze(-1)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Verify models
for name, Cls, kw in [
    ('GCN', GCN, {}),
    ('GIN', GIN, {}),
    ('GAT', GAT, {'heads': 4}),
    ('GraphSAGE', GraphSAGE, {}),
]:
    m = Cls(IN_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, **kw)
    n = count_params(m)
    print(f'{name}: {n:,} parameters')

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    pos_weight = torch.tensor([10.0], device=device)  # class weighting for 3.5% positives
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch.x.float(), batch.edge_index, batch.batch)
        labels = batch.y.float().view(-1)
        mask = ~torch.isnan(labels)
        loss = criterion(logits[mask], labels[mask])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * mask.sum().item()
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x.float(), batch.edge_index, batch.batch)
        labels = batch.y.float().view(-1)
        mask = ~torch.isnan(labels)
        probs = torch.sigmoid(logits[mask])
        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels[mask].cpu().numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    if labels.sum() == 0 or labels.sum() == len(labels):
        return 0.5  # edge case
    return roc_auc_score(labels, preds)


def train_model(model_cls, model_kwargs, name, train_loader, val_loader, test_loader,
                device, epochs=EPOCHS, patience=PATIENCE, lr=LR):
    model = model_cls(IN_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, **model_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=7, min_lr=1e-5
    )

    best_val_auc = 0
    best_test_auc = 0
    patience_count = 0
    history = {'train_loss': [], 'val_auc': [], 'test_auc': []}
    best_state = None
    
    print(f'\n{"="*50}')
    print(f'Training {name} | {count_params(model):,} params | {device}')
    print('='*50)
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, device)
        val_auc = evaluate(model, val_loader, device)
        test_auc = evaluate(model, test_loader, device)
        
        history['train_loss'].append(loss)
        history['val_auc'].append(val_auc)
        history['test_auc'].append(test_auc)
        
        scheduler.step(val_auc)
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_test_auc = test_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        if epoch % 10 == 0:
            elapsed = time.time() - t0
            print(f'Ep {epoch:3d} | loss={loss:.4f} | val_AUC={val_auc:.4f} | test_AUC={test_auc:.4f} | [{elapsed:.0f}s]')
        
        if patience_count >= patience:
            print(f'Early stopping at epoch {epoch} (patience={patience})')
            break
    
    elapsed = time.time() - t0
    print(f'\nBest val AUC: {best_val_auc:.4f} | Test AUC @ best val: {best_test_auc:.4f}')
    print(f'Total time: {elapsed:.0f}s')
    
    return {
        'name': name,
        'best_val_auc': best_val_auc,
        'best_test_auc': best_test_auc,
        'params': count_params(model),
        'epochs_run': len(history['val_auc']),
        'train_time_s': elapsed,
        'history': history,
        'best_state': best_state,
        'model_cls': model_cls,
        'model_kwargs': model_kwargs,
    }


print('Training utilities ready.')

## Experiment 2.1: GCN (Graph Convolutional Network)
**Hypothesis:** GCN should improve over fingerprints by modeling 1-3 hop molecular substructures.  
**Expected range:** 0.74–0.78 AUC (similar to or slightly below traditional ML due to simple aggregation)

In [ ]:
result_gcn = train_model(
    GCN, {}, 'GCN',
    train_loader, val_loader, test_loader, DEVICE
)
print(f'GCN Test AUC: {result_gcn["best_test_auc"]:.4f}')

## Experiment 2.2: GIN (Graph Isomorphism Network)
**Hypothesis:** GIN's MLP aggregation is strictly more powerful than GCN's mean-aggregation (Xu et al. proved this). Should outperform GCN.  
**Expected range:** 0.76–0.81 AUC. OGB's own GIN+virtual node reaches 0.7558 on this dataset.

In [ ]:
result_gin = train_model(
    GIN, {}, 'GIN',
    train_loader, val_loader, test_loader, DEVICE
)
print(f'GIN Test AUC: {result_gin["best_test_auc"]:.4f}')

## Experiment 2.3: GAT (Graph Attention Network)
**Hypothesis:** Attention mechanism allows the model to focus on chemically-relevant neighbors (e.g., aromatic bonds, functional groups). For molecules where specific substructures drive HIV activity, this should help.  
**Expected range:** 0.75–0.80 AUC.

In [ ]:
result_gat = train_model(
    GAT, {'heads': 4}, 'GAT',
    train_loader, val_loader, test_loader, DEVICE
)
print(f'GAT Test AUC: {result_gat["best_test_auc"]:.4f}')

## Experiment 2.4: GraphSAGE
**Hypothesis:** GraphSAGE's mean aggregation is fundamentally similar to GCN, but concatenation of self + neighbor embeddings preserves more information. Should perform between GCN and GIN.  
**Expected range:** 0.74–0.78 AUC.

In [ ]:
result_sage = train_model(
    GraphSAGE, {}, 'GraphSAGE',
    train_loader, val_loader, test_loader, DEVICE
)
print(f'GraphSAGE Test AUC: {result_sage["best_test_auc"]:.4f}')

In [ ]:
# ── Head-to-head comparison ──────────────────────────────────────────────
results = [result_gcn, result_gin, result_gat, result_sage]
results_sorted = sorted(results, key=lambda r: r['best_test_auc'], reverse=True)

# Phase 1 baselines (from PROGRESS_LOG)
phase1_baselines = [
    {'name': 'CatBoost (Mark, auto-weight)',  'best_test_auc': 0.7782, 'params': 'N/A', 'type': 'Traditional ML'},
    {'name': 'RF+Combined (Anthony)',          'best_test_auc': 0.7707, 'params': 'N/A', 'type': 'Traditional ML'},
    {'name': 'XGBoost+Combined',               'best_test_auc': 0.7613, 'params': 'N/A', 'type': 'Traditional ML'},
]
sota = {'name': 'DeeperGCN (SOTA)', 'best_test_auc': 0.8476, 'type': 'Published SOTA'}

print()
print('=' * 80)
print('HEAD-TO-HEAD COMPARISON — Phase 2 GNNs vs Phase 1 Traditional ML')
print('=' * 80)
header = '{:<5} {:<35} {:<12} {:<12} {:>12} {:>8}'.format('Rank', 'Model', 'Test AUC', 'Val AUC', 'Params', 'Time')
print(header)
print('-' * 80)
for rank, r in enumerate(results_sorted, 1):
    row = '{:<5} {:<35} {:.4f}      {:.4f}      {:>12,} {:>6.0f}s'.format(
        rank, r['name'], r['best_test_auc'], r['best_val_auc'], r['params'], r['train_time_s'])
    print(row)
print()
print('--- Phase 1 Traditional ML Baselines ---')
for b in phase1_baselines:
    row = '{:<5} {:<35} {:.4f}'.format('-', b['name'], b['best_test_auc'])
    print(row)
print()
print('--- Published SOTA ---')
print('{:<5} {:<35} {:.4f}'.format('-', sota['name'], sota['best_test_auc']))
print()

# Winner
winner = results_sorted[0]
gap_to_sota = sota['best_test_auc'] - winner['best_test_auc']
gap_to_cat  = 0.7782 - winner['best_test_auc']
print('Phase 2 Champion: {} (AUC={:.4f})'.format(winner['name'], winner['best_test_auc']))
print('Gap to SOTA DeeperGCN: {:+.4f}'.format(gap_to_sota))
print('Gap to Phase 1 CatBoost: {:+.4f}'.format(gap_to_cat))

In [ ]:
# ── Training curves plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4472C4', '#ED7D31', '#70AD47', '#FFC000']
model_names = ['GCN', 'GIN', 'GAT', 'GraphSAGE']
all_results = [result_gcn, result_gin, result_gat, result_sage]

for r, color, name in zip(all_results, colors, model_names):
    axes[0].plot(r['history']['val_auc'], label=f'{name} (val={r["best_val_auc"]:.4f})', 
                 color=color, linewidth=2)
    axes[1].plot(r['history']['train_loss'], label=name, color=color, linewidth=2)

axes[0].axhline(0.7782, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Phase 1 CatBoost (0.7782)')
axes[0].axhline(0.8476, color='red', linestyle=':', linewidth=2, alpha=0.7, label='SOTA DeeperGCN (0.8476)')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Val ROC-AUC', fontsize=12)
axes[0].set_title('Validation AUC — All GNN Architectures', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0.5, 0.90)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Training Loss', fontsize=12)
axes[1].set_title('Training Loss Curves', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'phase2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: phase2_training_curves.png')

In [ ]:
# ── Bar chart: all models compared ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

# All models in order
all_names  = [r['name'] for r in results_sorted] + [b['name'] for b in phase1_baselines] + [sota['name']]
all_aucs   = [r['best_test_auc'] for r in results_sorted] + [b['best_test_auc'] for b in phase1_baselines] + [sota['best_test_auc']]
all_colors = (
    ['#4472C4', '#4472C4', '#4472C4', '#4472C4'][:len(results_sorted)] +
    ['#70AD47'] * len(phase1_baselines) +
    ['#FF4444']
)

bars = ax.barh(all_names, all_aucs, color=all_colors, edgecolor='white', height=0.6, alpha=0.85)
for bar, auc in zip(bars, all_aucs):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{auc:.4f}', va='center', ha='left', fontsize=11, fontweight='bold')

ax.axvline(0.7782, color='#70AD47', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Test ROC-AUC', fontsize=12)
ax.set_title('Phase 2: GNN Architectures vs Traditional ML Baselines\nogbg-molhiv (scaffold split)', 
             fontsize=13, fontweight='bold')
ax.set_xlim(0.65, 0.92)
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4472C4', label='Phase 2 GNNs (this session)'),
    Patch(facecolor='#70AD47', label='Phase 1 Traditional ML'),
    Patch(facecolor='#FF4444', label='Published SOTA'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'phase2_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: phase2_model_comparison.png')

In [ ]:
# ── Per-model test prediction analysis ──────────────────────────────────
print('Running final predictions on test set...')
model_preds = {}
for r in all_results:
    m = r['model_cls'](IN_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, **r['model_kwargs']).to(DEVICE)
    m.load_state_dict({k: v.to(DEVICE) for k, v in r['best_state'].items()})
    m.eval()
    preds, labs = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            logits = m(batch.x.float(), batch.edge_index, batch.batch)
            labels = batch.y.float().view(-1)
            mask = ~torch.isnan(labels)
            probs = torch.sigmoid(logits[mask])
            preds.append(probs.cpu().numpy())
            labs.append(labels[mask].cpu().numpy())
    preds = np.concatenate(preds)
    labs = np.concatenate(labs)
    model_preds[r['name']] = {'probs': preds, 'labels': labs}

print()
print('Per-model performance on test set (threshold=0.5):')
header = '{:<20} {:<6} {:<6} {:<6} {:>12} {:>12} {:>8}'.format(
    'Model', 'TP', 'FP', 'FN', 'Sensitivity', 'Specificity', 'AUC')
print(header)
print('-' * 75)
for r in results_sorted:
    name = r['name']
    d = model_preds[name]
    pred_bin = (d['probs'] >= 0.5).astype(int)
    tp = int(((pred_bin == 1) & (d['labels'] == 1)).sum())
    fp = int(((pred_bin == 1) & (d['labels'] == 0)).sum())
    fn = int(((pred_bin == 0) & (d['labels'] == 1)).sum())
    tn = int(((pred_bin == 0) & (d['labels'] == 0)).sum())
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    row = '{:<20} {:<6} {:<6} {:<6} {:>12.3f} {:>12.3f} {:>8.4f}'.format(
        name, tp, fp, fn, sens, spec, r['best_test_auc'])
    print(row)

In [ ]:
# ── Save all results ─────────────────────────────────────────────────────
phase2_results = {}
for r in all_results:
    phase2_results[r['name']] = {
        'val_auc': r['best_val_auc'],
        'test_auc': r['best_test_auc'],
        'params': r['params'],
        'epochs_run': r['epochs_run'],
        'train_time_s': r['train_time_s'],
        'history': {
            'train_loss': r['history']['train_loss'],
            'val_auc': r['history']['val_auc'],
            'test_auc': r['history']['test_auc'],
        }
    }

# Load and update global metrics.json
metrics_path = RESULTS_DIR / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
else:
    metrics = {}

metrics['phase2_gnn_comparison'] = phase2_results
metrics['phase2_champion'] = {
    'model': results_sorted[0]['name'],
    'test_auc': results_sorted[0]['best_test_auc'],
    'val_auc': results_sorted[0]['best_val_auc'],
}
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

# Also save a detailed phase2 JSON
with open(RESULTS_DIR / 'phase2_gnn_results.json', 'w') as f:
    json.dump(phase2_results, f, indent=2)

print(f'Results saved to {RESULTS_DIR}')
print(f'\nPhase 2 Champion: {results_sorted[0]["name"]} — Test AUC = {results_sorted[0]["best_test_auc"]:.4f}')

## Key Findings — Phase 2

*(This section is completed after seeing actual results)*

### Summary Table (All Experiments to Date)

| Rank | Model | Type | Test AUC | Δ vs P1 Champion |
|------|-------|------|----------|------------------|
| — | DeeperGCN (SOTA) | Published | 0.8476 | +0.070 |
| P2.? | GIN | GNN | TBD | TBD |
| P2.? | GAT | GNN | TBD | TBD |
| P2.? | GCN | GNN | TBD | TBD |
| P2.? | GraphSAGE | GNN | TBD | TBD |
| P1.1 | CatBoost (Mark) | Trad. ML | 0.7782 | — |
| P1.2 | RF+Combined | Trad. ML | 0.7707 | -0.007 |
| P1.3 | XGBoost+Combined | Trad. ML | 0.7613 | -0.017 |

### Research Questions Answered

1. **Do simple GNNs beat traditional ML on molecular graph classification?** → See results above
2. **Which GNN architecture is most powerful?** → GIN (theoretically) vs. empirical results
3. **How far are we from SOTA?** → Gap narrows in Phase 3 with advanced architectures

### Next Steps (Phase 3)
- Deep-dive on Phase 2 champion GNN architecture
- Add edge features (bond type, ring membership, aromaticity) — currently unused
- Test virtual node addition (OGB's technique for long-range interactions)
- Implement residual/skip connections for deeper networks
- Try GIN+JK (Jumping Knowledge) aggregation across all layers